<a href="https://colab.research.google.com/github/Alien-73/Building_Energy_Flex/blob/main/build_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building Simulation

In [1]:
pip install casadi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 9.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from scipy.linalg import expm
import casadi
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import clear_output

clear_output()

In [9]:
nx = 2 #The number of states
nu = 1 #The number of controls
#-----------------------------------------

#Cost function weights (?)
Q = casadi.diag([1,0]) #The weights as in Q[0]*(x[0]-x_ref) + Q[1]*(x[1]-x_ref) + ...
R = casadi.diag([0.1,0.5])  #The weights as in R[0]*(u[0]-u_ref) + R[1]*(u[1]-u_ref) + ...
Q_f = casadi.diag([1,1]) #?

#constraints
x_lb = [20,-np.inf] #20 #Additon: I can delete these to make the soft constraint on temp.
x_ub = [24,np.inf] #22
u_lb = [0,0]#
u_ub = [6,5]#

#traget values
x_ref = casadi.DM([21,0]) #ADDITION: I only put reference on x_0 (T_in). and T_e we dont care. Change for temp. 21deg.C for example
u_ref = casadi.DM([0,0]) #ADDITION: Keep 0 because we rather not use heat pump to save vost

total = nx*(K+1) + nu*K #Dimentions of optimization variables


In [29]:
#Inputs
#----------------------------------------------
#Building Model:

#Type: Residential
#Building model type: 3R2C (Ref: DTU summer course) -> Rie, Rea, Rinf, Ci, Ce
#Parameters
Rea = 58.4895  #['C/kW]
Rie = 1.1521  #['C/kW]
Rinf = 15.748 #['C/kW]
Ce = 4.22     #[kWh/'C]
Ci = 1.15     #[kWh/'C]
Ai = 1.56     #[m^2]
Ae = 0.122    #[m^2]
#----------------------------------------------
#Weather and and External datasets
# reading csv file
df = pd.read_csv("data.csv")
#print(df.head())

#Qint = df["Qint"]#"Qint":internal process heating [kW]
#Qvent = df["Qvent"]#"Qvent":the ventilation heating loss [kW]
Gh = df["Gv"] #"Gh" horizental solar radiation [kW/m.sq]
Ta = df["Ta"] #"Ta" ourdoor temp. [de.C]
#This is unknown #"Q_h":heating input [kW]

# Define T_o_seri and Q_sol_seri from the loaded data
T_o_seri = Ta
Q_sol_seri = Gh

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#Weather noise
p11 = -2.89726 #FROM DTU case - Summer course Friday session
p22 = -0.06793 #FROM DTU case - Summer course Friday session
#Noise
Noise_level = 0.5
#generated based on Brownian Motion Noise
np.random.seed(seed=45)
#noise_Ti = np.cumsum(np.random.normal(0,sd_Ti , t_eval.size)) #sd_Ti*np.sqrt(dt)
#noise_Th = np.cumsum(np.random.normal(0, sd_Th*np.sqrt(dt), t_eval.size))
#noise_Te = np.cumsum(np.random.normal(0,sd_Tm , t_eval.size)) #sd_Te*np.sqrt(dt)

#noise_Ti = Noise_level* np.diff(noise_Ti,axis=0)
#noise_Te = Noise_level* np.diff(noise_Te,axis=0)
#OR pre calculated noise fro CSV
df2 = pd.read_csv('noise.csv')
noise_Ti = df2["Bin"]* Noise_level
noise_Te = df2["Bm"]* Noise_level
noise_Ti = np.diff(noise_Ti,axis=0)
noise_Te = np.diff(noise_Te,axis=0)
#----------------------------------------
#Simulation Inputs:
t_sim = 75 #Total Simulation horizon [hr]
T = 6     #Control Horizon [hr] -> Number of hr to predict (in my case 6 x 10-min steps which is 1hr)
K = 24    #Control Horizon [Steps]
dt = T/K  #Sampling time [hr] (for me 15 mins)

Steps_sim = t_sim/dt+1 #Number of steps [-]
#Generating timeline
t_eval = np.arange(0,t_sim+T,dt)
#----------------------------------------
#Prices
#Fixed price:
el_price = int(Steps_sim) * [0.909]
#X = [Ti Te]*T
#U = [Q_h Qvent Qint Ta Gh]*T  #"Qint" "Qvent" "Gh" "Ta" are known
#OR FROM CSV
#df3 = pd.read_csv('price.csv')
#-------------------------------------
#Heating System
Eff_h = 0.95 #Eff. of the  heating in the building - ATT: should be changed if HP is being used

x_init = casadi.DM([23.5,23.5]) #initial values
x0 = casadi.DM.zeros(total)

In [30]:
#Functions & preprocessing

#Building
#Ordinary Diff. Equation in the following form
#___________________________________________________________________
#dTi/dt = (1/(Ci*Rie)*(Te-Ti) + 1/(Ci*Rinf)*(Ta-Ti) + 1/Ci*(Qh + Qint(x) + Qvent(Already known, I am gonna use them!) + Qdh + Ai*Gh))*dt + dw11*dt
#dTe/dt = (1/(Ce*Rie)*(Ti-Te) + 1/(Ce*Rea)*(Ta-Te) + 1/Ce*(Ae*Gh)*dt + exp(p22)/Cm*dw22)
#___________________________________________________________________

import numpy as np
from scipy.linalg import expm
#Descretsizing the continuous state-space model:
#dx/dt  = A*x + B*u
#y      = C*x + D*u
#to:
#x(k+1) = Ad*x + Bd*u
#y(k)   = C*x + D*u

def discretize(A, B, dT): #A and B as above and T is the timestep (0.25hr)
    # Compute A_d
    A_d = expm(A * dT)

    # Compute B_d using the formula B_d = A_inv * (A_d - I) * B
    # if A is invertible
    n = A.shape[0]
    B_d = np.linalg.inv(A) @ (A_d - np.eye(n)) @ B

    return A_d, B_d

def AB_matrices(Rea, Rie, Rinf, Ci, Ce,Ai,Ae, eta): #System model
    """
    Build continuous-time A (2x2) and B (2x2) matrices for the 3R2C thermal model:
      [Ti_dot, Te_dot]^T = A [Ti, Te]^T + B [Qh, Qinit, Qvent,T_out, Q_sol]^T
    """
    # Safety valve --> divide-by-zero; small floor
    eps = 1e-9
    Rea  = max(Rea,  eps)
    Rie  = max(Rie,  eps)
    Rinf = max(Rinf, eps)
    Ci   = max(Ci,   eps)
    Ce   = max(Ce,   eps)

    #A (2x2)#
    A = np.array([[-1/(Ci*Rie)-1/(Ci*Rinf), 1/(Ci*Rie)], [1/(Ce*Rie), -1/(Ce*Rie)-1/(Ce*Rea)]])
    #B (2x5)#
    B = np.array([[ 1/Ci, 1/(Ci*Rinf) , Ai/Ci], [0 , 1/(Ce*Rea) , Ae/Ce]])
    #A = np.array([[-(1/(Ci*Rie) + 1/(Ci*Rinf)), (1/(Ci*Rie))],
    #             [(1/(Ce*Rie)), -(1/(Ce*Rie) + 1/(Ce*Rea))]])

    #B = np.array([[1/(Ci*Rinf),  eta/Ci],
    #              [1/(Ce*Rea) ,  0.0]])
    return A, B

A, B = AB_matrices(Rea, Rie, Rinf, Ci, Ce,Ai,Ae, Eff_h)
Ad, Bd = discretize(A, B, dt)

In [31]:
#State_Equations
def make_F(T_o = 0, Q_sol = 0,n_Ti=0,n_Te=0):
  states = casadi.SX.sym("states",nx)
  ctrls = casadi.SX.sym("ctrls",nu) #
  Ti = states[0]
  Te = states[1]
  Qh = ctrls[0]

  #state-space equations - Discrete
  #EQ: X(n+1) = A*X(n) + B*U(n) + Brownian_Motion_noise #,
  [Ti_next,Te_next] = np.dot(A,np.array([Ti,Te])) + np.dot(B,np.array([Qh*Eff_h,T_o,Q_sol])) + np.hstack((n_Ti,n_Te)) #

  states_next = casadi.vertcat(Ti_next,Te_next)
  F = casadi.Function("F",[states,ctrls],[states_next],["x","u"],["x_next"])
  return F

In [32]:
#Cost function and MPC
def compute_stage_cost(x,u,t): #,w1
  x_diff = casadi.fabs(x[0] - x_ref[0]) #ADDITION: It tries to keep state close to the ref 1.0 at the end. This is where I can add the cost function for deviation of tempetrature
  u_diff = u - u_ref
  cost =   el_price*casadi.dot(R@u_diff,u_diff) #5*(casadi.dot(Q@x_diff,x_diff)) +
  #print (x_diff)

  return cost #, comfort

def compute_terminal_cost(x):
  x_diff = casadi.fabs(x - x_ref)
  cost = casadi.dot(Q_f@x_diff,x_diff)/2
  return cost

#Optimization Problem
def make_nlp(t2=0): #w3,
  F = make_F(T_o=Ta[t2],Q_sol=Gh[t2]) #

  U = [casadi.SX.sym(f"u_{k}",nu) for k in range (K)]
  X = [casadi.SX.sym(f"x_{k}",nx) for k in range (K+1)]
  G = []

  J = 0

  for k in range(K):
    J += compute_stage_cost(X[k],U[k],t=t2+k)[0] #,w1=y2
    eq = X[k+1] - F(x=X[k],u=U[k])["x_next"]
    G.append(eq)
  J += compute_terminal_cost(X[-1])

  option = {"print_time":False, "ipopt":{"print_level":0}}
  nlp = {"x":casadi.vertcat(*X,*U),"f":J,"g":casadi.vertcat(*G)}
  S = casadi.nlpsol("S","ipopt",nlp,option)
  return S

In [34]:
#"Optimal Control calculation"
def compute_optimal_control(S,x_init,x0):
  x_init = x_init.full().ravel().tolist()

  lbx = x_init + x_lb*K + u_lb*K
  ubx = x_init + x_ub*K + u_ub*K
  lbg = [0]*nx*K
  ubg = [0]*nx*K

  res = S(lbx=lbx,ubx=ubx,lbg=lbg,ubg=ubg, x0=x0)

  offset = nx*(K+1)
  x0 = res["x"]
  u_opt = x0[offset:offset+nu]

  u_opt = np.round(u_opt/0.5)*0.5 #to round the u_opt to integer numbers
  #u_opt[1] = np.round(u_opt[1]/0.5)*0.5
  return u_opt, x0

In [35]:
obj = dict()
X = [x_init]
U = []
x_current = x_init
for t in range(t_eval.size-2*K):
    S = make_nlp(t2=t) #y2=y3,
    F = make_F(T_o=Ta[t],Q_sol=Gh[t],n_Ti=noise_Ti[t],n_Te=noise_Te[t]) #Adding this T_o_seri[t] Q_sol_seri[t] noise_Ti[t] noise_Te[t]
    if t%6 == 0:   #ADDITION: This if statement shows how fast "u" can be changed. Now it can take a new variable every 6*1/6 =1 h
      u_opt, x0 = compute_optimal_control(S,x_current,x0)
    x_current = F(x=x_current,u=u_opt)["x_next"]
    X.append(x_current)
    U.append(u_opt)

RuntimeError: Error in Function::call for 'S' [IpoptInterface] at .../casadi/core/function.cpp:1466:
Error in Function::call for 'S' [IpoptInterface] at .../casadi/core/function.cpp:362:
.../casadi/core/function_internal.hpp:1728: Input 2 (lbx) has mismatching shape. Got 98-by-1. Allowed dimensions, in general, are:
 - The input dimension N-by-M (here 74-by-1)
 - A scalar, i.e. 1-by-1
 - M-by-N if N=1 or M=1 (i.e. a transposed vector)
 - N-by-M1 if K*M1=M for some K (argument repeated horizontally)
 - N-by-P*M, indicating evaluation with multiple arguments (P must be a multiple of 1 for consistency with previous inputs)